In [1]:
!pip install -q -U langgraph langchain langchain-groq

In [2]:
import os
from getpass import getpass

os.environ.pop("GROQ_API_KEY", None)

# Securely enter Groq API key
groq_api_key = getpass("Enter your Groq API Key: ").strip()

if not groq_api_key:
    raise ValueError("Groq API key cannot be empty!")

os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API configured successfully!")

Enter your Groq API Key: ··········
Groq API configured successfully!


In [3]:
from langchain_core.tools import tool


@tool
def addition(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


@tool
def subtraction(a: float, b: float) -> float:
    """Subtract the second number from the first number."""
    return a - b


@tool
def multiplication(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b


@tool
def division(a: float, b: float) -> str:
    """Divide the first number by the second number."""

    if b == 0:
        return "Division by zero is not allowed."

    return str(a / b)

tools = [
    addition,
    subtraction,
    multiplication,
    division
]

print("Tools registered successfully!")

for item in tools:
    print(f"{item.name}: {item.description}")

Tools registered successfully!
addition: Add two numbers and return the result.
subtraction: Subtract the second number from the first number.
multiplication: Multiply two numbers and return the result.
division: Divide the first number by the second number.


In [4]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    max_retries=1,
    timeout=45
)

print("Testing Groq connection...")

test_response = llm.invoke(
    "Reply with: Groq connection successful."
)

print("Model response:", test_response.content)

# Create LangGraph agent
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are an intelligent and helpful AI assistant.

    Your responsibilities:

    1. Understand the user's questions.
    2. Answer general knowledge questions directly.
    3. Explain technical concepts clearly with examples.
    4. Use mathematical tools whenever calculations are required.
    5. Select the appropriate tool based on the user's request.
    6. Analyze tool outputs before generating the final answer.
    7. Maintain conversational context.
    8. Always provide a meaningful final response.

    Do not use tools unnecessarily.
    Answer in clear and understandable language.
    """
)

print("\nLangGraph Single Agent created successfully!")

Testing Groq connection...
Model response: Groq connection successful.

LangGraph Single Agent created successfully!


In [5]:
from IPython.display import display, Markdown

print("TEST 1: General Question")

result1 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain database design in simple words."
        }
    ]
})

print("\nAgent Response:")
print(result1["messages"][-1].content)


print("\n\nTEST 2: Mathematical Calculation")

result2 = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 250 multiplied by 40?"
        }
    ]
})

print("\nAgent Response:")
print(result2["messages"][-1].content)


print("\n\nLANGGRAPH WORKFLOW")

graph = agent.get_graph()

mermaid_diagram = graph.draw_mermaid()

display(Markdown(
    f"```mermaid\n{mermaid_diagram}\n```"
))

TEST 1: General Question

Agent Response:
**Database design** is like planning a house before you build it.  
Just as you decide how many rooms, where the doors go, and how the plumbing will run, database design decides:

| What you’re building | What it means in a database |
|----------------------|-----------------------------|
| **Tables (rooms)** | Separate “rooms” for different kinds of data (e.g., *Customers*, *Orders*, *Products*). |
| **Columns (walls)** | The attributes of each room (e.g., *CustomerID*, *Name*, *Address*). |
| **Rows (people inside rooms)** | Individual records, one for each customer, order, or product. |
| **Keys (doorways)** | Special columns that uniquely identify a row (*primary key*) or link rows together (*foreign key*). |
| **Relationships (hallways)** | How tables connect, like a customer “has” many orders. |
| **Constraints (rules)** | Rules that keep the data tidy, e.g., “email must be unique” or “price can’t be negative.” |
| **Indexes (quick‑access

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools<hr/><small><em>__tools = addition, subtraction, multiplication, division</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Initialize conversation history
conversation_history = []

print("=" * 60)
print("             GROQ LANGGRAPH SINGLE AGENT")
print("=" * 60)

print("Hello! I am your AI assistant.")
print("Ask me anything!")

print("\nCommands:")
print("exit  - Stop the agent")
print("reset - Clear conversation history")

print("=" * 60)


while True:

    user_input = input("\nYou: ").strip()


    if user_input.lower() == "exit":
        print("\nAgent: Goodbye! Have a nice day.")
        break


    if user_input.lower() == "reset":
        conversation_history = []
        print("\nAgent: Conversation history cleared!")
        continue

    if not user_input:
        continue

    try:

        print("\nAgent is thinking...")


        current_messages = conversation_history + [
            HumanMessage(content=user_input)
        ]


        result = agent.invoke(
            {
                "messages": current_messages
            },
            config={
                "recursion_limit": 10
            }
        )


        conversation_history = result["messages"]

        final_message = next(
            (
                message
                for message in reversed(result["messages"])
                if isinstance(message, AIMessage)
                and not message.tool_calls
                and message.content
            ),
            None
        )

        if final_message:

            print("\nAgent:")

            answer = final_message.content

            if isinstance(answer, str):
                print(answer)

            elif isinstance(answer, list):
                for item in answer:
                    if isinstance(item, dict):
                        if item.get("type") == "text":
                            print(item.get("text", ""))

        else:
            print("\nAgent: No final response was generated.")

    except Exception as error:
        print("\nAgent Error:", str(error))

             GROQ LANGGRAPH SINGLE AGENT
Hello! I am your AI assistant.
Ask me anything!

Commands:
exit  - Stop the agent
reset - Clear conversation history

You: hi

Agent is thinking...

Agent:
Hello! How can I help you today?

You: what is interface Interface Optimization

Agent is thinking...

Agent:
**Interface Optimization** is a broad concept that refers to the systematic improvement of a “boundary” or “touchpoint” where two systems, components, or people interact.  The goal is to make that interaction as efficient, reliable, and user‑friendly as possible.  Depending on the domain, the term can take slightly different flavors:

| Domain | Typical meaning of “interface” | What is being optimized | Common goals |
|--------|--------------------------------|------------------------|--------------|
| **Software / UI** | User interface (buttons, menus, screens) | Usability, speed, error‑rate | Faster task completion, lower cognitive load, higher satisfaction |
| **Hardware / Embedded